# makemore — a bigram character-level language model

Cleaned-up notes from Karpathy's *Zero to Hero*, the first makemore video.

---

### What we're building

A **character-level language model**: it treats each character as an example and learns to predict the *next* character in a sequence. Train it on a list of names and it will invent new name-like strings, one character at a time.

We build the *same* model twice, which is the whole point of the video:

1. **By counting** — tally how often each character follows each other character, turn the tallies into probabilities, and sample.
2. **As a neural network** — learn those same probabilities with gradient descent.

The punchline: they compute the identical thing. The counts become the network's learned weights.

## 0. The data

In [ ]:
words = open('../data/names.txt', 'r').read().splitlines()

In [ ]:
print(words[:10])
print(len(words))
print(min(len(w) for w in words))
print(max(len(w) for w in words))

---
## 1. Counting bigrams

A **bigram** model looks only at the *previous* character to predict the next one. That's a weak language model — it has almost no context — but it's the simplest thing that works.

Each name is padded so the model can learn how names *start* and *end*. The video first uses separate `<S>`/`<E>` tokens, then simplifies to a single `.` token for both — that's what the rest of the notebook uses.

In [ ]:
# first pass: count bigrams into a dict, using <S>/<E> start/end markers.
# only looking at the previous character makes this a weak (bigram) model.
b = {}
for w in words:
    chs = ['<S>'] + list(w) + ['<E>']
    for ch1, ch2 in zip(chs, chs[1:]):
        bigram = (ch1, ch2)
        b[bigram] = b.get(bigram, 0) + 1

In [ ]:
# most frequent bigrams
sorted(b.items(), key=lambda kv: -kv[1])[:10]

### The count matrix `N`

Instead of a dict, store the counts in a 27×27 tensor: 26 letters plus one `.` token (index 0) that marks both start and end. `stoi`/`itos` map characters to indices and back.

In [ ]:
import torch

In [ ]:
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0                     # single token for both start and end
itos = {i: s for s, i in stoi.items()}

N = torch.zeros((27, 27), dtype=torch.int32)
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        N[stoi[ch1], stoi[ch2]] += 1

### Visualizing the counts

Row `i` = the current character, column `j` = the next character; the cell is how often `j` follows `i`.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=(16, 16))
plt.imshow(N, cmap='Blues')
for i in range(27):
    for j in range(27):
        chstr = itos[i] + itos[j]
        plt.text(j, i, chstr, ha="center", va="bottom", color='gray')
        plt.text(j, i, N[i, j].item(), ha="center", va="top", color='gray')
plt.axis('off');

### From counts to probabilities

Take one row (all the characters that can follow `.`) and normalize it so it sums to 1 — now it's a probability distribution. `torch.multinomial` then samples a character from it.

In [ ]:
N[0]

In [ ]:
p = N[0].float()
p = p / p.sum()
p

In [ ]:
g = torch.Generator().manual_seed(2147483647)
ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
itos[ix]

### A quick multinomial sanity check

`multinomial` just draws indices in proportion to the probabilities you hand it.

In [ ]:
g = torch.Generator().manual_seed(2147483647)
p = torch.rand(3, generator=g)
p = p / p.sum()
p

In [ ]:
torch.multinomial(p, num_samples=100, replacement=True, generator=g)

### Normalizing the whole matrix at once — and broadcasting

Rather than normalizing a row at a time, normalize every row of `N` in one shot.

**Broadcasting** is how PyTorch lets a `(27, 27)` matrix be divided by a `(27, 1)` column of row-sums: the column is *virtually copied* across all 27 columns to match the matrix's shape, then the division happens element-wise. No data is actually duplicated — it just behaves as if it were.

**The dangerous part:** `keepdims=True`. It keeps the sum as shape `(27, 1)` so it broadcasts down each row. Drop it and you get shape `(27,)`, which broadcasts across the *wrong* axis and silently normalizes columns instead of rows — a classic, hard-to-spot bug.

The `(N + 1)` is **model smoothing**: adding one fake count to every bigram removes zeros, so no bigram has probability 0 (which would make its log-likelihood infinite).

In [ ]:
g = torch.Generator().manual_seed(2147483647)

P = (N + 1).float()               # +1 smoothing: no zero probabilities
P /= P.sum(1, keepdims=True)      # broadcasting; keepdims=True is essential (see above)

# sample 15 names from the count model
for i in range(15):
    out = []
    ix = 0
    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0:
            break
    print(''.join(out))

---
## 2. How good is the model? Negative log likelihood

We need a single number that says how well the model explains the data.

- **Likelihood** = the product of the probabilities the model assigns to every bigram that actually occurred. Higher is better, but multiplying thousands of small numbers underflows to zero.
- **Log likelihood** = the sum of the *logs* of those probabilities (`log(a*b*c) = log a + log b + log c`). Same ranking, but a stable sum instead of a vanishing product.
- **Negative log likelihood (NLL)** = `-log likelihood`. We flip the sign because we want a **loss** — lower = better — and log likelihood runs the other way (higher = better).
- **Average NLL** = NLL divided by the number of bigrams. A cleaner, dataset-size-independent number. This is the loss we minimize.

The best possible loss is 0 (the model was perfectly certain and correct every time).

In [ ]:
log_likelihood = 0.0
n = 0

for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        prob = P[stoi[ch1], stoi[ch2]]
        logprob = torch.log(prob)
        log_likelihood += logprob
        n += 1

print(f'{log_likelihood=}')
nll = -log_likelihood
print(f'{nll=}')
print(f'average nll = {nll / n}')

---
## 3. The same model as a neural network

Now rebuild the bigram model as a tiny network, so it can be *trained* instead of *counted*.

First, the training set: every bigram becomes one example, input `x` (current character index) → label `y` (next character index).

In [ ]:
# build the (x, y) training set from the first word, to see the shape of things
xs, ys = [], []
for w in words[:1]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        print(ch1, ch2)
        xs.append(stoi[ch1])
        ys.append(stoi[ch2])

xs = torch.tensor(xs)
ys = torch.tensor(ys)

In [ ]:
xs

In [ ]:
ys

### One-hot encoding

A character index isn't meaningful as a number (index 5 isn't "half of" index 10). **One-hot encoding** turns each index into a length-27 vector that is 1 in that character's slot and 0 everywhere else — a clean input for a linear layer.

In [ ]:
import torch.nn.functional as F
xenc = F.one_hot(xs, num_classes=27).float()
xenc

In [ ]:
xenc.shape

In [ ]:
plt.imshow(xenc)

### The forward pass: logits → counts → probabilities

`@` is PyTorch's **matrix-multiplication** operator. Multiplying the one-hot input by a weight matrix `W` (27×27) produces 27 numbers per example.

Here's the counts ↔ logits ↔ softmax picture that ties the two models together:

- In the **count** model we had `N` (counts) → normalize → probabilities.
- The network outputs **logits** = *log-counts*. Exponentiating, `logits.exp()`, gives **counts** — positive numbers playing exactly the role of `N`. Normalizing those gives probabilities.
- "**exponentiate, then normalize**" *is* the **softmax** function.

So a logit is just the network's learned stand-in for the log of a count. `W` is the learnable version of (the log of) the count matrix — instead of tallying `N`, gradient descent will discover it.

In [ ]:
W = torch.randn((27, 27))
xenc @ W

In [ ]:
# forward pass
logits = xenc @ W                          # log-counts
counts = logits.exp()                      # counts, equivalent to N
probs = counts / counts.sum(1, keepdims=True)   # softmax = exp then normalize
print(probs[0])
print(probs[0].shape)
print(probs[0].sum())                      # each row sums to 1

### Reading off the loss by hand

For each of the first 5 bigrams: what probability did the (untrained, random) net assign to the *correct* next character? Its negative log is that example's loss; the average is the batch loss.

In [ ]:
nlls = torch.zeros(5)
for i in range(5):
    x = xs[i].item()   # input character index
    y = ys[i].item()   # label character index
    print('--------')
    print(f'bigram example {i + 1}: {itos[x]}{itos[y]} (indexes {x},{y})')
    print('input to the neural net:', x)
    print('output probabilities from the neural net:', probs[i])
    print('label (actual next character):', y)
    p = probs[i, y]
    print('prob assigned to the correct character:', p.item())
    logp = torch.log(p)
    print('log likelihood:', logp.item())
    nll = -logp
    print('negative log likelihood:', nll.item())
    nlls[i] = nll

print('=========')
print('average negative log likelihood, i.e. loss =', nlls.mean().item())

---
## 4. Optimization (training)

**A forward pass alone is prediction. Training is the forward pass plus a backward pass through a loss function** — backprop tells each weight how it affected the loss, and we step against the gradient.

One full step on the 5-example toy set: forward → loss → `backward()` → nudge `W`. (Note `W.grad = None` first — clearing gradients before each backward pass, since they accumulate.)

In [ ]:
print(xs)
print(ys)

# randomly initialize the 27x27 weights; requires_grad so we can backprop
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

# forward pass
xenc = F.one_hot(xs, num_classes=27).float()      # one-hot input
logits = xenc @ W                                  # log-counts
counts = logits.exp()                              # counts (~ N)
probs = counts / counts.sum(1, keepdims=True)      # softmax
loss = -probs[torch.arange(5), ys].log().mean()    # average NLL
print(loss.item())

# backward pass
W.grad = None                                      # clear gradients first
loss.backward()

# update
W.data += -0.1 * W.grad

### Train on the full dataset

Build every bigram in the data, then run gradient descent.

**Regularization.** The extra `+ 0.01 * (W**2).mean()` pushes the weights toward 0. Small weights → logits near 0 → counts near 1 → *uniform* probabilities, so it's a smoothing pressure (the gradient-based twin of the `+1` we added to `N` earlier). The `0.01` controls how much regularization — too much and every prediction blurs toward uniform.

The learning rate here (`-50`) is large because this loss surface is simple and well-behaved. In deeper models you typically **decay the learning rate** — start with big steps and shrink them at higher epochs, so you move fast early and settle precisely near the minimum.

In [ ]:
# full training set
xs, ys = [], []
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        xs.append(stoi[ch1])
        ys.append(stoi[ch2])
xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print('number of examples:', num)

# initialize the 'network'
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

In [ ]:
# gradient descent
for k in range(100):
    # forward pass
    xenc = F.one_hot(xs, num_classes=27).float()
    logits = xenc @ W                                  # log-counts
    counts = logits.exp()                              # counts (~ N)
    probs = counts / counts.sum(1, keepdims=True)      # softmax
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01 * (W**2).mean()   # NLL + L2 reg
    print(loss.item())

    # backward pass
    W.grad = None                                      # clear gradients first
    loss.backward()

    # update
    W.data += -50 * W.grad                             # step against the gradient

---
## 5. Sampling from the trained network

Same sampling loop as the count model, but now each next-character distribution comes from the network's forward pass (one-hot → `@ W` → softmax). The output should be indistinguishable from the count model's — because, trained, it *is* the same model.

In [ ]:
g = torch.Generator().manual_seed(2147483647)

for i in range(5):
    out = []
    ix = 0
    while True:
        xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
        logits = xenc @ W
        counts = logits.exp()
        p = counts / counts.sum(1, keepdims=True)      # softmax

        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0:
            break
    print(''.join(out))

---
## Recap

| concept | one line |
|---|---|
| **character language model** | predicts the next character; each character is a training example |
| **bigram** | uses only the previous character as context — weak but simple |
| **broadcasting** | virtually copy a smaller tensor to match a larger one, then operate element-wise (`keepdims=True` matters!) |
| **`@`** | matrix multiplication in PyTorch |
| **one-hot encoding** | index → length-27 vector, 1 in the character's slot |
| **logits** | log-counts — the network's output before softmax |
| **softmax** | `exp(logits)` then normalize → probabilities |
| **likelihood → log-likelihood → NLL** | product of probs → sum of log-probs → negate for a loss (lower = better) |
| **average NLL** | NLL / number of examples — the clean loss to minimize |
| **forward vs backward** | forward = prediction; forward + loss + backward = training |
| **regularization** | `+λ·mean(W²)` pushes weights toward 0 → smoother, more uniform predictions |
| **learning-rate decay** | shrink the step size at higher epochs to settle near the minimum |
